# 03 — Pilot Run (24 samples)

A 4-per-task sanity check. Runs Gemma-3n-E2B-IT on stages S2-S4 only —
just enough to confirm: model loads, prompts elicit the bracket format,
parsing works, predictions look sensible.

**Runtime:** ~5-10 min on Colab L4.


In [ ]:
# ─── Colab bootstrap ────────────────────────────────────────
# Mount Drive (caches model weights + videos across sessions),
# clone the repo, install dependencies, and configure the cache dirs.
import os, sys, subprocess, pathlib

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/avut"
    REPO_DIR = "/content/omnimodel-research"
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    if not os.path.exists(REPO_DIR):
        subprocess.run([
            "git", "clone",
            "https://github.com/samadasyed/omnimodel-research.git",
            REPO_DIR,
        ], check=True)
else:
    DRIVE_ROOT = os.path.expanduser("~/avut")
    REPO_DIR = str(pathlib.Path.cwd())
    os.makedirs(DRIVE_ROOT, exist_ok=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Persistent storage layout (data + model caches under DRIVE_ROOT)
DATA_DIR        = os.path.join(DRIVE_ROOT, "data")
VIDEO_DIR       = os.path.join(DATA_DIR, "videos")
AUDIO_DIR       = os.path.join(DATA_DIR, "audio")
SILENT_DIR      = os.path.join(DATA_DIR, "silent")
TRANSCRIPT_DIR  = os.path.join(DATA_DIR, "transcripts")
ANNOTATION_DIR  = os.path.join(DATA_DIR, "annotations")
RESULTS_DIR     = os.path.join(DRIVE_ROOT, "results")
RAW_PRED_DIR    = os.path.join(RESULTS_DIR, "raw_predictions")
METRICS_DIR     = os.path.join(RESULTS_DIR, "metrics")
HF_CACHE        = os.path.join(DRIVE_ROOT, ".cache", "hf")
WHISPER_CACHE   = os.path.join(DRIVE_ROOT, ".cache", "whisper")

for d in [VIDEO_DIR, AUDIO_DIR, SILENT_DIR, TRANSCRIPT_DIR,
          ANNOTATION_DIR, RAW_PRED_DIR, METRICS_DIR, HF_CACHE, WHISPER_CACHE]:
    os.makedirs(d, exist_ok=True)

# Redirect HF cache to Drive so we don't redownload weights each session
os.environ["HF_HOME"] = HF_CACHE
os.environ["TRANSFORMERS_CACHE"] = HF_CACHE
os.environ["HF_DATASETS_CACHE"] = HF_CACHE

print(f"Repo:       {REPO_DIR}")
print(f"Drive root: {DRIVE_ROOT}")
print(f"HF cache:   {HF_CACHE}")


In [ ]:
# ─── Install model dependencies ──────────────────────────
# transformers (Gemma-3n preview support landed in 4.51), accelerate,
# soundfile for audio I/O, openai-whisper for ASR. cv2 (opencv-python)
# is preinstalled on Colab and used for frame sampling.
%pip install -q -U "transformers>=4.51.0" "accelerate>=0.30"     "huggingface-hub>=0.24" "soundfile>=0.12" scipy
%pip install -q openai-whisper
# Qwen-Omni preview branch is only needed for notebook 05; install there.

# Gemma-3n is GATED on HuggingFace — you must accept the license at
# https://huggingface.co/google/gemma-3n-E2B-it and authenticate.
# In Colab: Tools → Secrets → add HF_TOKEN, then:
import os
if "HF_TOKEN" in os.environ or "HUGGINGFACE_TOKEN" in os.environ:
    from huggingface_hub import login
    login(token=os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN"))
    print("HF authenticated")
else:
    try:
        from google.colab import userdata
        tok = userdata.get("HF_TOKEN")
        if tok:
            from huggingface_hub import login
            login(token=tok)
            print("HF authenticated (from Colab Secrets)")
        else:
            print("WARN: no HF_TOKEN — Gemma-3n download will fail. "
                  "Add HF_TOKEN to Colab Secrets first.")
    except Exception:
        print("WARN: not in Colab and no HF_TOKEN env var. "
              "Run `huggingface-cli login` before loading Gemma.")


In [ ]:
import json
from src import data_utils, stages
from src.model_utils import Gemma3nWrapper

manifest_path = os.path.join(DATA_DIR, "sample_manifest_available.json")
with open(manifest_path) as f:
    full_manifest = json.load(f)

# 4 per task → 24 samples
pilot = data_utils.balanced_subsample(full_manifest, n_per_task=4, seed=7)
print(f"Pilot samples: {len(pilot)}")


In [ ]:
# Load Gemma-3n
gemma = Gemma3nWrapper(model_name="google/gemma-3n-E2B-it")
print("Gemma loaded")


In [ ]:
# Run a few stages on the pilot — just to verify everything works
def get_paths(s):
    vid = s["video_id"]
    return {
        "video":  os.path.join(VIDEO_DIR, f"{vid}.mp4"),
        "silent": os.path.join(SILENT_DIR, f"{vid}_silent.mp4"),
        "audio":  os.path.join(AUDIO_DIR, f"{vid}.wav"),
        "transcript": os.path.join(TRANSCRIPT_DIR, f"{vid}.txt"),
    }

PILOT_STAGES = ["S2_audio_only", "S3_visual_only", "S4_full_av"]
pilot_results = {s: [] for s in PILOT_STAGES}

for stage in PILOT_STAGES:
    fn = stages.STAGE_REGISTRY[stage]["fn"]
    for s in pilot:
        try:
            row = fn(gemma, s, get_paths(s))
            pilot_results[stage].append(row)
        except Exception as e:
            print(f"  ✗ {stage} qa_id={s['qa_id']}: {e}")
    valid = sum(1 for r in pilot_results[stage] if r.get("predicted_answer"))
    print(f"  {stage}: {valid}/{len(pilot)} parseable")


In [ ]:
# Quick accuracy summary
from src import metrics
acc = metrics.compute_accuracy(pilot_results)
for stage, per_task in acc.items():
    print(f"\n{stage}:")
    for task, d in per_task.items():
        if d.get("accuracy") is not None:
            print(f"  {task:35s} {d['accuracy']:.2f} ({d['n_correct']}/{d['n_valid']})")


If everything looks right above, proceed to `04_main_eval_gemma.ipynb`.